# Shadow Slave full-novel index (Colab)

Builds the bge-large retrieval index for all 3,127 chapters and zips it for Weaver.
One-time job. Run the cells **in order**.

**Before you start:**
1. Runtime -> Change runtime type -> **T4 GPU** -> Save.
2. Upload **`shadow-slave.zip`** (made locally with `zip -r shadow-slave.zip novels/shadow-slave`) via the files panel.

**No notebook updates needed, ever.** The script and question set are downloaded fresh from GitHub
in cell 3, so fixes reach you by just re-running cell 3. Never re-upload this notebook.

**After cell 1 you must see** `IMPORTS OK:` with `CUDAExecutionProvider` and a GPU name on the torch line.

In [ ]:
# 1. Install dependencies with ONLY the GPU onnxruntime, CUDA 12 build.
#    Colab's T4 is CUDA 12; the default onnxruntime-gpu (1.28+) is a CUDA 13
#    build and fails to load. The CUDA 12 build lives on Microsoft's feed.
!pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null | tail -1
!pip install -q fastembed qdrant-client
!pip uninstall -y onnxruntime 2>/dev/null | tail -1
!pip install -q onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/
!python -c "import fastembed, qdrant_client, onnxruntime as ort; print('IMPORTS OK:', ort.__version__, ort.get_available_providers())"
!python -c "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - Runtime > Change runtime type > T4 GPU')"
!ls novels/shadow-slave | wc -l   # 33 = novel present; 0 = re-upload the zip and re-run cell 2

In [ ]:
# 2. Unzip the novel and sanity-check the layout
!unzip -q shadow-slave.zip
!ls novels/shadow-slave | tail -3
!ls novels/shadow-slave | wc -l   # expect 33 rows: 32 chapter dirs + urls.md

In [ ]:
# 3. Fetch the LATEST script and question set from GitHub (no re-uploads, ever)
!curl -sfL https://raw.githubusercontent.com/haxsysgit/Weaver/main/scripts/colab_index.py -o colab_index.py && wc -l colab_index.py
!curl -sfL https://raw.githubusercontent.com/haxsysgit/Weaver/main/scripts/questions-colab.json -o questions-colab.json && wc -c questions-colab.json

## 4. Chunk-size sweep

Builds a temp index per size, scores the 4 novel arms on the 35 questions, deletes each temp index.
**Scope: chapters 1-1000** (all 35 questions live there). The final build in step 5 covers all 3,127 chapters.

The script prints `dense providers: [...]` - it must include CUDAExecutionProvider.
Expect roughly **5-10 minutes per size** on a working T4.

The headline number is the **dense hit@5** row - pick the size with the best dense hit@5
(ties -> prefer the smaller chunk). Read the table below the run.

In [ ]:
# 4. Sweep chunk sizes: 12, 20, 30, 40 over chapters 1-1000 (~5-10 min each on T4)
!python colab_index.py --novels-dir novels/shadow-slave \
    --questions questions-colab.json --sweep 12,20,30,40 \
    --max-chapters 1000 \
    --report /content/report.md
!cat /content/report.md

## 5. Build the real index

After the sweep, set the winning size here and run this cell. It builds the
full-novel index (all 3,127 chapters) at that size and zips it (~100-200 MB download).
Expect **~15-25 minutes** on the T4.

In [ ]:
# 5. Build at the winning size over ALL 3,127 chapters, then zip
CHUNK_TARGET = 20   # <- winning size from the sweep
CHUNK_OVERLAP = 8   # <- max(1, CHUNK_TARGET // 2 - 1)
!python colab_index.py --novels-dir novels/shadow-slave \
    --chunk-target {CHUNK_TARGET} --chunk-overlap {CHUNK_OVERLAP} \
    --report /content/build-report.md --zip-out /content/index.zip
!ls -lh /content/index.zip

In [ ]:
# 6. Download the index to your machine
from google.colab import files
files.download('/content/index.zip')
files.download('/content/report.md')
files.download('/content/build-report.md')

## Back on your machine

```bash
cd /home/hax/weaver
rm -rf .weaver/retrieval/index
unzip -q ~/Downloads/index.zip -d .weaver/retrieval/

uv run python - <<'PY'
from qdrant_client import QdrantClient
c = QdrantClient(path=".weaver/retrieval/index")
info = c.get_collection("novel_chunks")
print("points:", info.points_count, "dense dim:", info.config.params.vectors["dense"].size)
PY
```

Expect ~22k points, dense dim 1024 (bge-large). The runtime loads this index lazily on first search.